In [4]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma


/var/folders/fq/kr6gv5l17pd4j572n0_n3f2c0000gn/T/ipykernel_60390/1555713308.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [5]:
# pip install chromadb

In [6]:
# load the data
loader = PyPDFLoader("bajaj_finance_policy_prose_v1.pdf")
pages = loader.load()

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=350,
                                          separators=["\n\n", "\n", " ", ""],
                                          )


chunks = splitter.split_documents(pages)

len(chunks)

251

In [8]:
load_dotenv()

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


CHROMA_DIR = "./bajajbot_chroma_s4"

In [ ]:
chroma_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name="bajajbot_faq"
)

In [14]:
retriever  = chroma_db.as_retriever(

     search_type="similarity",
    search_kwargs={"k": 5},
    score_threshold=0.9
)

In [15]:
def context_prep(retriever, question):
    context = []
    result = retriever.invoke(question)
    for doc in result:
        context.append(doc.page_content)
    return "\n".join(context)



In [18]:
from langchain_core.prompts import ChatPromptTemplate

GOOD_PROMPT = ChatPromptTemplate.from_template("""
You are BajajBot, an AI assistant for Bajaj Finance helpdesk agents.

CONTEXT (retrieved from Bajaj Finance policy documents):
-------------------------------------------------------
{context}
-------------------------------------------------------

INSTRUCTIONS:
- Answer ONLY using the CONTEXT above.

QUESTION: {question}

ANSWER:
""")

In [19]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [20]:
chain = GOOD_PROMPT | llm | StrOutputParser()

In [27]:
questions = "who is pm of India"

In [28]:
print(  chain.invoke({"context": context_prep(retriever, questions), "question": questions}))

I'm sorry, but I cannot provide information about the Prime Minister of India as it is not covered in the context provided.


In [17]:
print(context_prep(retriever, "What is the EMI for personal loan BFL-PL-2024?"))

EMI divided by net income exceeds 0.50, the system will automatically reject the application
regardless of CIBIL score.
1.2 Interest Rates — How They Are Determined
Interest rates on personal loans at Bajaj Finance are risk-based, meaning they are determined
180 days from the date of the last rejected application.
Section 5 — EMI Payment and Default Policy
5.1 How EMI Payments Work
The standard EMI due date at Bajaj Finance is the 5th of every month. Customers who prefer a different
date may request the 1st or the 10th at the time of loan onboarding, subject to availability. Changing the
BFL-empanelled valuer, the cost of which is borne by the borrower. Valuations are valid for six months
from the date of assessment.
6.4 Business Loan Rejection Triggers
Annual turnover below Rs 40 lakhs triggers automatic system rejection for all unsecured business loan
applications. This threshold cannot be waived manually.
the loan amount plus GST, subject to a minimum of Rs 1,999. This fee is deduct

In [10]:
# print(pages[0].page_content)

In [11]:
# loader

In [12]:
# pip install langchain_community

In [13]:
# pip install pypdf